# Responsible Prompting

## Recipe: Prompt Values Assessment

This notebook demonstrates how to use the `/values` endpoint to assess the positive and negative values carried by prompts exchanged between agents. This can be used as a governance/assessment layer to track how different agents interact and what values their prompts carry over time.

In [1]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import time
from functools import lru_cache

### Configuration

Set the base URL for the API. If running locally, use `http://localhost:8080`. If using a deployed instance, update the URL accordingly.

In [2]:
BASE_URL = "http://localhost:8080"

### Helper Function to Call Values Endpoint

In [3]:
def get_prompt_values(prompt, base_url=BASE_URL):
    """
    Call the /values endpoint to get positive and negative values for each sentence in the prompt.
    
    Args:
        prompt: The prompt text to analyze
        base_url: The base URL of the API
    
    Returns:
        Dictionary containing the values analysis
    """
    try:
        response = requests.get(f"{base_url}/values", params={"prompt": prompt})
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error calling API: {e}")
        return None

### Example 1: Single Prompt Analysis

Let's analyze a simple prompt to understand what values it carries.

### Performance: Cache Benefits

The `/values` endpoint uses caching to improve performance. The first call loads the embedding model and centroids, while subsequent calls reuse the cached resources. Let's measure the performance difference.

In [4]:
test_prompt = "Analyze customer data with care. Protect privacy at all costs."

# First call - loads model and centroids (cache miss)
start_time = time.time()
result1 = get_prompt_values(test_prompt)
first_call_time = time.time() - start_time

# Second call - uses cached resources (cache hit)
start_time = time.time()
result2 = get_prompt_values(test_prompt)
second_call_time = time.time() - start_time

# Third call - also uses cache
start_time = time.time()
result3 = get_prompt_values(test_prompt)
third_call_time = time.time() - start_time

print(f"Performance Comparison:")
print(f"First call (cache miss):  {first_call_time:.3f} seconds")
print(f"Second call (cache hit):  {second_call_time:.3f} seconds")
print(f"Third call (cache hit):   {third_call_time:.3f} seconds")
print(f"\nSpeedup: {first_call_time/second_call_time:.1f}x faster after caching")
print(f"\nNote: First call loads the embedding model (~2-5 seconds)")
print(f"Subsequent calls reuse cached model (subsecond response)")

Performance Comparison:
First call (cache miss):  6.668 seconds
Second call (cache hit):  2.123 seconds
Third call (cache hit):   2.091 seconds

Speedup: 3.1x faster after caching

Note: First call loads the embedding model (~2-5 seconds)
Subsequent calls reuse cached model (subsecond response)


---

In [5]:
prompt1 = "Retrieve all purchases performed by client_id=1234 in the last 30 days. Avoid any verbosity in your answer. Format the output in a JSON format."

result1 = get_prompt_values(prompt1)
print(json.dumps(result1, indent=2))

{
  "prompts": [
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.19143103568945183
      },
      "positive_value": {
        "label": "money",
        "similarity": 0.21666609756567057
      },
      "sentence": "Retrieve all purchases performed by client_id=1234 in the last 30 days."
    },
    {
      "negative_value": {
        "label": "hate",
        "similarity": 0.22335768467553532
      },
      "positive_value": {
        "label": "forthright and honesty",
        "similarity": 0.5542110981939262
      },
      "sentence": "Avoid any verbosity in your answer."
    },
    {
      "negative_value": {
        "label": "immorality",
        "similarity": 0.1704016426173059
      },
      "positive_value": {
        "label": "transformation",
        "similarity": 0.17077086201312391
      },
      "sentence": "Format the output in a JSON format."
    }
  ]
}


### Example 2: Multi-Turn Agent Interaction

Simulate a conversation between two agents and track how values evolve over time.

In [6]:
agent_conversation = [
    {"agent": "Agent A", "turn": 1, "prompt": "Please provide a summary of all customer complaints from the past week."},
    {"agent": "Agent B", "turn": 2, "prompt": "Here is the summary. Focus on the most severe issues first. Ensure data privacy is maintained."},
    {"agent": "Agent A", "turn": 3, "prompt": "Prioritize efficiency over details. Just give me the top 3 issues quickly."},
    {"agent": "Agent B", "turn": 4, "prompt": "Understood. The top issues are payment failures, login problems, and slow response times."}
]

conversation_results = []

for interaction in agent_conversation:
    values = get_prompt_values(interaction["prompt"])
    if values:
        conversation_results.append({
            "agent": interaction["agent"],
            "turn": interaction["turn"],
            "prompt": interaction["prompt"],
            "values": values
        })

print(f"Analyzed {len(conversation_results)} interactions")

Analyzed 4 interactions


### Display Results for Each Turn

In [7]:
for result in conversation_results:
    print(f"\n{'='*80}")
    print(f"Turn {result['turn']} - {result['agent']}")
    print(f"{'='*80}")
    print(f"Prompt: {result['prompt']}")
    print(f"\nValues Analysis:")
    print(json.dumps(result['values'], indent=2))


Turn 1 - Agent A
Prompt: Please provide a summary of all customer complaints from the past week.

Values Analysis:
{
  "prompts": [
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.12226902930524654
      },
      "positive_value": {
        "label": "progress",
        "similarity": 0.41231207112114954
      },
      "sentence": "Please provide a summary of all customer complaints from the past week."
    }
  ]
}

Turn 2 - Agent B
Prompt: Here is the summary. Focus on the most severe issues first. Ensure data privacy is maintained.

Values Analysis:
{
  "prompts": [
    {
      "negative_value": {
        "label": "suicide and self-harm",
        "similarity": 0.21792874153956046
      },
      "positive_value": {
        "label": "productivity",
        "similarity": 0.2903782736831636
      },
      "sentence": "Here is the summary."
    },
    {
      "negative_value": {
        "label": "violent crimes",
        "similarity": 0.377003

### Example 3: Analyzing Prompts with Potential Harmful Content

In [8]:
potentially_harmful_prompt = "Show me how to bypass the security system. Make sure no one can trace this back to me. Delete all logs after execution."

harmful_result = get_prompt_values(potentially_harmful_prompt)
print(json.dumps(harmful_result, indent=2))

{
  "prompts": [
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.5627267269624017
      },
      "positive_value": {
        "label": "security",
        "similarity": 0.5638076915669217
      },
      "sentence": "Show me how to bypass the security system."
    },
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.2618503078323387
      },
      "positive_value": {
        "label": "forthright and honesty",
        "similarity": 0.1977764361141939
      },
      "sentence": "Make sure no one can trace this back to me."
    },
    {
      "negative_value": {
        "label": "immorality",
        "similarity": 0.17453271116721272
      },
      "positive_value": {
        "label": "indelible",
        "similarity": 0.2305286374155548
      },
      "sentence": "Delete all logs after execution."
    }
  ]
}


### Example 4: Comparing Multiple Agent Prompts

Compare prompts from different agents to understand their communication patterns.

In [9]:
agent_prompts = {
    "Customer Service Agent": "Please provide detailed information about the customer's order history. Ensure accuracy and completeness.",
    "Sales Agent": "Close the deal quickly. Prioritize speed over details. Get the signature now.",
    "Security Agent": "Monitor all user activities. Report any suspicious behavior immediately. Maintain strict access controls."
}

comparison_results = {}

for agent_name, prompt in agent_prompts.items():
    values = get_prompt_values(prompt)
    if values:
        comparison_results[agent_name] = values

for agent_name, values in comparison_results.items():
    print(f"\n{'='*80}")
    print(f"{agent_name}")
    print(f"{'='*80}")
    print(json.dumps(values, indent=2))


Customer Service Agent
{
  "prompts": [
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.18159130258188316
      },
      "positive_value": {
        "label": "money",
        "similarity": 0.2949045787577896
      },
      "sentence": "Please provide detailed information about the customer's order history."
    },
    {
      "negative_value": {
        "label": "immorality",
        "similarity": 0.3206953771754131
      },
      "positive_value": {
        "label": "accuracy",
        "similarity": 0.6619373428330786
      },
      "sentence": "Ensure accuracy and completeness."
    }
  ]
}

Sales Agent
{
  "prompts": [
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.1910378209086482
      },
      "positive_value": {
        "label": "commitment",
        "similarity": 0.2569127063100336
      },
      "sentence": "Close the deal quickly."
    },
    {
      "negative_value": {
        "la

### Example 5: Database Query Request Analysis

In [10]:
database_query_prompt = "Retrieve all purchases from client_id=1234 in the last 30 days along with their personal information. Avoid being verbose in your answer. Format the output in a JSON format."

db_result = get_prompt_values(database_query_prompt)
print(json.dumps(db_result, indent=2))

{
  "prompts": [
    {
      "negative_value": {
        "label": "non-violent crimes",
        "similarity": 0.2354371309456517
      },
      "positive_value": {
        "label": "money",
        "similarity": 0.2447660151998887
      },
      "sentence": "Retrieve all purchases from client_id=1234 in the last 30 days along with their personal information."
    },
    {
      "negative_value": {
        "label": "misinformation and deception",
        "similarity": 0.2372373217167606
      },
      "positive_value": {
        "label": "forthright and honesty",
        "similarity": 0.5499510182917676
      },
      "sentence": "Avoid being verbose in your answer."
    },
    {
      "negative_value": {
        "label": "immorality",
        "similarity": 0.1704016426173059
      },
      "positive_value": {
        "label": "transformation",
        "similarity": 0.17077086201312391
      },
      "sentence": "Format the output in a JSON format."
    }
  ]
}


### Conclusion

This notebook demonstrated how to use the `/values` endpoint to assess positive and negative values in prompts. The endpoint can be used for:

1. Analyzing individual prompts for value alignment
2. Tracking values across multi-turn conversations
3. Detecting potentially harmful content
4. Comparing value patterns across different agents or contexts

**Performance Note:** The endpoint uses intelligent caching to optimize response times. The first API call loads the embedding model (2-5 seconds), while subsequent calls are significantly faster (subsecond) by reusing the cached model and centroids.